In [14]:
import os
import numpy as np
import pandas as pd
import os
import json
import scipy.stats as stats
from pathlib import Path
import re

os.chdir(os.getcwd())

## Walk Over Results


In [ ]:
dir_path = Path('./results/')

In [ ]:
def process_metrics_jsons(directory_path):
    """
    Process JSON files containing training metrics and combine them into a DataFrame.
    Each JSON file contains lists of metrics across epochs.

    Parameters:
    directory_path (str): Path to the directory containing JSON files

    Returns:
    pandas.DataFrame: Combined DataFrame with metrics and their sources
    """
    # Convert directory path to Path object
    dir_path = Path(directory_path)

    # List to store all data
    all_data = []

    # Walk through directory
    for file_path in dir_path.rglob('*.json'):
        try:
            # Read JSON file
            with open(file_path, 'r') as file:
                data = json.load(file)

            # Get the number of epochs (length of any metric list)
            n_epochs = len(data['balanced_accuracy'])

            # Create records for each epoch
            for epoch in range(n_epochs):
                record = {
                    'source_file': file_path.name,
                    'epoch': epoch,
                    'balanced_accuracy': data['balanced_accuracy'][epoch],
                    'val_loss': data['val_loss'][epoch],
                    'val_f1': data['val_f1'][epoch],
                    'val_AUPRC': data['val_AUPRC'][epoch],
                    'val_MCC': data['val_MCC'][epoch]
                }
                all_data.append(record)

        except Exception as e:
            print(f"Error processing {file_path.name}: {str(e)}")
            continue

    if not all_data:
        raise ValueError("No valid JSON files found in the directory")

    # Convert to DataFrame
    df = pd.DataFrame(all_data)

    # Reorder columns for better readability
    column_order = ['source_file', 'epoch', 'balanced_accuracy', 'val_loss',
                    'val_f1', 'val_AUPRC', 'val_MCC']
    df = df[column_order]

    print(
        f"Processed {len(set(df['source_file']))} files with {len(df)} total epochs")
    return df

In [206]:

def parse_filename(filename):
    """
    Parse a complex filename and extract different components into a dictionary.

    Args:
    filename (str): The input filename to parse

    Returns:
    dict: A dictionary containing parsed components
    """
    # Remove .json extension if present
    filename = filename.replace('.json', '')

    # Split the filename by underscores
    parts = filename.split('_')

    # Initialize a dictionary to store parsed values
    parsed_dict = {}

    # Extract dataset name (first part)
    parsed_dict['dataset_name'] = parts[0]

    # Create a dictionary to store metric values
    metric_values = {}

    if 'no-polyak' in filename:
        parsed_dict['polyak'] = False
    elif 'polyak' in filename:
        parsed_dict['polyak'] = True
    parsed_dict['is_baseline'] = False
    for model in ["logistic_regression", "random_forest", "gradient_boosting", "knn"]:
        if model in filename:
            parsed_dict['is_baseline'] = True

    # Process remaining parts
    for part in parts[1:]:
        # Split each part into metric and value
        metric_match = re.match(r'([a-z]+)(?:-(\d+(?:over\d+)?))?', part)

        if metric_match:
            metric = metric_match.group(1)
            value = metric_match.group(2) if metric_match.group(2) else '0'

            # Convert fractional values like '1over2' to float
            if 'over' in str(value):
                num, denom = value.split('over')
                value = f'{num} / {denom}'
            else:
                value = str(value)

            metric_values[metric] = value

    # Add specific metrics to the dictionary
    metrics_to_extract = ['euclidean', 'chebyshev', 'wasserstein']
    for metric in metrics_to_extract:
        parsed_dict[metric] = metric_values.get(metric, 0)

    return parsed_dict


def parse_filename_column(df, filename_column):
    """
    Apply filename parsing to an entire DataFrame column.

    Args:
    df (pd.DataFrame): Input DataFrame
    filename_column (str): Name of the column containing filenames

    Returns:
    pd.DataFrame: DataFrame with parsed columns added
    """
    # Apply parsing to each filename
    parsed_data = df[filename_column].apply(parse_filename)

    # Convert parsed data to DataFrame
    parsed_df = pd.DataFrame(parsed_data.tolist())

    # Combine original DataFrame with parsed columns
    return pd.concat([df, parsed_df], axis=1)

In [207]:
dfs = process_metrics_jsons(dir_path)

Processed 54 files with 2160 total epochs


In [208]:
aggregated_df = dfs.groupby('source_file').mean().reset_index()

In [209]:
aggregated_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   source_file        54 non-null     object 
 1   epoch              54 non-null     float64
 2   balanced_accuracy  54 non-null     float64
 3   val_loss           54 non-null     float64
 4   val_f1             54 non-null     float64
 5   val_AUPRC          54 non-null     float64
 6   val_MCC            54 non-null     float64
dtypes: float64(6), object(1)
memory usage: 3.1+ KB


In [210]:
aggregated_df.head(10)

,source_file,epoch,balanced_accuracy,val_loss,val_f1,val_AUPRC,val_MCC
0,CICEVSE_Network2024_gradient_boosting.json,19.5,0.497249,0.000000,0.052343,0.104050,-0.052245
1,CICEVSE_Network2024_knn.json,19.5,0.724911,0.000000,0.425803,0.375872,0.871447
2,CICEVSE_Network2024_logistic_regression.json,19.5,0.481233,0.000000,0.017051,0.071098,-0.067375
3,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,19.5,0.796456,0.684253,0.782712,0.640831,0.745302
4,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,19.5,0.789633,0.675779,0.806688,0.668950,0.769522
5,CICEVSE_Network2024_no-polyak_euclidean-1over3...,19.5,0.819206,0.674375,0.850751,0.733309,0.818949
6,CICEVSE_Network2024_no-polyak_euclidean-1over4...,19.5,0.807802,0.685952,0.809966,0.670989,0.771867
7,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,19.5,0.772876,0.726851,0.742132,0.577184,0.692637
8,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,19.5,0.801906,0.711774,0.805844,0.664820,0.766504
9,CICEVSE_Network2024_polyak_euclidean-1over3_ch...,19.5,0.814350,0.712984,0.835176,0.708854,0.801173


In [211]:
parsed_df = parse_filename_column(aggregated_df, 'source_file')

### Baseline Results

In [212]:
baseline_df = parsed_df[parsed_df['is_baseline'] == True]

In [213]:
baseline_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16 entries, 0 to 53
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   source_file        16 non-null     object 
 1   epoch              16 non-null     float64
 2   balanced_accuracy  16 non-null     float64
 3   val_loss           16 non-null     float64
 4   val_f1             16 non-null     float64
 5   val_AUPRC          16 non-null     float64
 6   val_MCC            16 non-null     float64
 7   dataset_name       16 non-null     object 
 8   is_baseline        16 non-null     bool   
 9   euclidean          16 non-null     object 
 10  chebyshev          16 non-null     object 
 11  wasserstein        16 non-null     object 
 12  polyak             0 non-null      object 
dtypes: bool(1), float64(6), object(6)
memory usage: 1.6+ KB


In [214]:
baseline_table_df = baseline_df.sort_values('balanced_accuracy', ascending=False)\
    .drop(columns=['is_baseline', 'polyak', 'val_loss', 'euclidean', 'chebyshev', 'wasserstein'])\
    .sort_values(['dataset_name', 'balanced_accuracy'], ascending=[True, False]) \
    .groupby('dataset_name') \
    .head(1)

In [215]:
baseline_table_df

,source_file,epoch,balanced_accuracy,val_f1,val_AUPRC,val_MCC,dataset_name
11,CICEVSE_Network2024_random_forest.json,19.5,0.821173,0.640275,0.567085,0.941916,CICEVSE
24,CICIDS2017_gradient_boosting.json,19.5,0.721114,0.407909,0.364668,0.934306,CICIDS2017
42,CICIoV2024_logistic_regression.json,19.5,0.979838,0.962288,0.962450,0.990591,CICIoV2024


In [240]:
ordered_baseline = baseline_table_df[['dataset_name', 'source_file', 'balanced_accuracy', 'val_f1', 'val_AUPRC', 'val_MCC']].copy()

### Polyak stuff

In [216]:
results = parsed_df[parsed_df['is_baseline'] == False]

In [217]:
results.head()

,source_file,epoch,balanced_accuracy,val_loss,val_f1,val_AUPRC,val_MCC,dataset_name,is_baseline,euclidean,chebyshev,wasserstein,polyak
3,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,19.5,0.796456,0.684253,0.782712,0.640831,0.745302,CICEVSE,False,0,1 / 2,1 / 2,False
4,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,19.5,0.789633,0.675779,0.806688,0.668950,0.769522,CICEVSE,False,0,1 / 3,1 / 3,False
5,CICEVSE_Network2024_no-polyak_euclidean-1over3...,19.5,0.819206,0.674375,0.850751,0.733309,0.818949,CICEVSE,False,1 / 3,1 / 3,0,False
6,CICEVSE_Network2024_no-polyak_euclidean-1over4...,19.5,0.807802,0.685952,0.809966,0.670989,0.771867,CICEVSE,False,1 / 4,1 / 4,1 / 4,False
7,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,19.5,0.772876,0.726851,0.742132,0.577184,0.692637,CICEVSE,False,0,1 / 2,1 / 2,True


In [225]:
results_table = results.sort_values(['dataset_name','balanced_accuracy'], ascending=[True, False]) \
    .groupby(['dataset_name', 'polyak']) \
    .head(1) \
    .drop(columns=['is_baseline', 'epoch'])

In [231]:
column_order =[
 'dataset_name',
 'polyak',
 'balanced_accuracy',
 'val_f1',
 'val_AUPRC',
 'val_MCC',
 'euclidean',
 'chebyshev',
 'wasserstein'
]

In [229]:
list(results_table.columns)

['source_file',
 'balanced_accuracy',
 'val_loss',
 'val_f1',
 'val_AUPRC',
 'val_MCC',
 'dataset_name',
 'euclidean',
 'chebyshev',
 'wasserstein',
 'polyak']

In [230]:
results_table

,source_file,balanced_accuracy,val_loss,val_f1,val_AUPRC,val_MCC,dataset_name,euclidean,chebyshev,wasserstein,polyak
5,CICEVSE_Network2024_no-polyak_euclidean-1over3...,0.819206,0.674375,0.850751,0.733309,0.818949,CICEVSE,1 / 3,1 / 3,0,False
9,CICEVSE_Network2024_polyak_euclidean-1over3_ch...,0.814350,0.712984,0.835176,0.708854,0.801173,CICEVSE,1 / 3,1 / 3,0,True
36,CICIDS2017_polyak_euclidean-1over2_chebyshev-0...,0.876787,0.714908,0.662575,0.465844,0.634678,CICIDS2017,1 / 2,0,0,True
31,CICIDS2017_no-polyak_euclidean-1over3_chebyshe...,0.873976,0.678629,0.672710,0.479766,0.641218,CICIDS2017,1 / 3,1 / 3,0,False
47,CICIoV2024_no-polyak_euclidean-1over4_chebyshe...,0.981283,0.619619,0.971847,0.950642,0.913599,CICIoV2024,1 / 4,1 / 4,1 / 4,False
52,CICIoV2024_polyak_euclidean-1over4_chebyshev-1...,0.971816,0.700572,0.940362,0.899166,0.847008,CICIoV2024,1 / 4,1 / 4,1 / 4,True


In [234]:
ordered_results = results_table[column_order].copy()

## LATEX Stuff


Utils for updating the LaTex tables dynamically


In [145]:
def dataframe_to_latex_escape(df, caption="", label="", column_format=None, fit_single_column=False):
    """
    Converts a Pandas DataFrame to a LaTeX table, escaping special characters and ignoring the index.

    Parameters:
        df (pd.DataFrame): The DataFrame to convert.
        caption (str): The caption for the table.
        label (str): The label for the table (used for referencing in LaTeX).
        column_format (str): Optional LaTeX column format string (e.g., "lccccccc").
        fit_single_column (bool): If True, resize the table to fit a single column.

    Returns:
        str: The LaTeX string for the table.
    """
    # Escape underscores and other LaTeX special characters in column names
    df.columns = [col.replace('_', r'\_') for col in df.columns]

    # Escape underscores and other special characters in data
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: str(x).replace('_', r'\_'))

    # Define default column alignment
    if column_format is None:
        column_format = "l" + "c" * (df.shape[1] - 1)

    # Convert the DataFrame to a LaTeX table without the index
    latex_table = df.to_latex(
        index=False,  # Ignore the index
        header=True,
        column_format=column_format,
        float_format="{:.4f}".format,
        escape=False  # Disable escape as we are escaping manually
    )

    # Wrap table with resizing if required
    if fit_single_column:
        latex_table = (
            "\\begin{table}[h]\n"
            "\\centering\n"
            "\\resizebox{\\columnwidth}{!}{%\n"
            + latex_table +
            "}\n"
            f"\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n"
            "\\end{table}"
        )
    else:
        latex_table = (
            "\\begin{table*}[h]\n"
            "\\centering\n"
            + latex_table +
            f"\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n"
            "\\end{table*}"
        )

    return latex_table

### Latex for resuts

In [242]:
# Or to print directly
latex_output = dataframe_to_latex_escape(ordered_results)
print(latex_output)

\begin{table*}[h]
\centering
\begin{tabular}{lcccccccc}
\toprule
dataset\\_name & polyak & balanced\\_accuracy & val\\_f1 & val\\_AUPRC & val\\_MCC & euclidean & chebyshev & wasserstein \\
\midrule
CICEVSE & False & 0.8192 & 0.8508 & 0.7333 & 0.8189 & 1 / 3 & 1 / 3 & 0 \\
CICEVSE & True & 0.8143 & 0.8352 & 0.7089 & 0.8012 & 1 / 3 & 1 / 3 & 0 \\
CICIDS2017 & True & 0.8768 & 0.6626 & 0.4658 & 0.6347 & 1 / 2 & 0 & 0 \\
CICIDS2017 & False & 0.8740 & 0.6727 & 0.4798 & 0.6412 & 1 / 3 & 1 / 3 & 0 \\
CICIoV2024 & False & 0.9813 & 0.9718 & 0.9506 & 0.9136 & 1 / 4 & 1 / 4 & 1 / 4 \\
CICIoV2024 & True & 0.9718 & 0.9404 & 0.8992 & 0.8470 & 1 / 4 & 1 / 4 & 1 / 4 \\
\bottomrule
\end{tabular}
\caption{}
\label{}
\end{table*}


### Baseline

In [243]:
latex_output = dataframe_to_latex_escape(ordered_baseline)
print(latex_output)

\begin{table*}[h]
\centering
\begin{tabular}{lccccc}
\toprule
dataset\\_name & source\\_file & balanced\\_accuracy & val\\_f1 & val\\_AUPRC & val\\_MCC \\
\midrule
CICEVSE & CICEVSE\\_Network2024\\_random\\_forest.json & 0.8212 & 0.6403 & 0.5671 & 0.9419 \\
CICIDS2017 & CICIDS2017\\_gradient\\_boosting.json & 0.7211 & 0.4079 & 0.3647 & 0.9343 \\
CICIoV2024 & CICIoV2024\\_logistic\\_regression.json & 0.9798 & 0.9623 & 0.9625 & 0.9906 \\
\bottomrule
\end{tabular}
\caption{}
\label{}
\end{table*}


## CICIoV


In [45]:
def read_csv_files_in_folder(folder_path):
    """
    Reads CSV files from a folder and concatenates them into a single pandas DataFrame.

    Parameters:
    folder_path (str): Path to the folder containing the CSV files

    Returns:
    pandas.DataFrame: Concatenated DataFrame from all CSV files in the folder
    """
    # Get a list of all CSV files in the folder
    csv_files = [os.path.join(folder_path, f)
                 for f in os.listdir(folder_path) if f.endswith('.csv')]

    # Read each CSV file and append to a list of DataFrames
    dfs = []
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, low_memory=False)
        df['target'] = df['specific_class'].str.lower()
        dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    return combined_df

In [34]:
data = pd.read_parquet('./data/parquets/cicevse_network.parquet')

In [35]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482309 entries, 0 to 482308
Data columns (total 88 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            482309 non-null  int64  
 1   expiration_id                 482309 non-null  int64  
 2   src_ip                        482309 non-null  object 
 3   src_mac                       482309 non-null  object 
 4   src_oui                       482309 non-null  object 
 5   src_port                      482309 non-null  int64  
 6   dst_ip                        482309 non-null  object 
 7   dst_mac                       482309 non-null  object 
 8   dst_oui                       482309 non-null  object 
 9   dst_port                      482309 non-null  int64  
 10  protocol                      482309 non-null  int64  
 11  ip_version                    482309 non-null  int64  
 12  vlan_id                       482309 non-nul

In [ ]:
data.head().iloc[:, 10:]

,protocol,ip_version,vlan_id,tunnel_id,bidirectional_first_seen_ms,bidirectional_last_seen_ms,bidirectional_duration_ms,bidirectional_packets,bidirectional_bytes,src2dst_first_seen_ms,...,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,target,state
0,6,4,0,0,1703261991666,1703261991779,113,2,120,1703261991666,...,Unspecified,0,0,None,None,None,None,None,aggressive-scan,charging
1,6,4,0,0,1703261991667,1703261991784,117,2,120,1703261991667,...,Email,1,1,None,None,None,None,None,aggressive-scan,charging
2,6,4,0,0,1703261991667,1703261991784,117,2,120,1703261991667,...,System,1,1,None,None,None,None,None,aggressive-scan,charging
3,6,4,0,0,1703261991667,1703261991785,118,2,120,1703261991667,...,Email,1,1,None,None,None,None,None,aggressive-scan,charging
4,6,4,0,0,1703261991667,1703261991785,118,2,120,1703261991667,...,RPC,1,1,None,None,None,None,None,aggressive-scan,charging


In [ ]:
def read_ciciov(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1).iloc[:, 1:-3]
    y = df[target]
    return x, y

In [ ]:
def read_cicevse(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1).iloc[:, 14:-6]
    y = df[target]
    return x, y

In [46]:
def read_data(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1)
    y = df[target]
    return x, y

In [ ]:
def preprocess_data(X, y, features, batch_size):
    scaler = sklearn.preprocessing.RobustScaler()

    X_scaled = scaler.fit_transform(X[features].values)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.float32)
    n_features = X_scaled.shape[1]

    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=train_batch_size, shuffle=True)

    return X_tensor, y_tensor, dataloader

In [3]:
X, Y = read_ciciov('./data/parquets/ciciov.parquet', 'target')

In [4]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1408219 entries, 0 to 1408218
Data columns (total 8 columns):
 #   Column  Non-Null Count    Dtype
---  ------  --------------    -----
 0   DATA_0  1408219 non-null  int64
 1   DATA_1  1408219 non-null  int64
 2   DATA_2  1408219 non-null  int64
 3   DATA_3  1408219 non-null  int64
 4   DATA_4  1408219 non-null  int64
 5   DATA_5  1408219 non-null  int64
 6   DATA_6  1408219 non-null  int64
 7   DATA_7  1408219 non-null  int64
dtypes: int64(8)
memory usage: 86.0 MB


In [58]:
Y

0                  benign
1                  benign
2                  benign
3                  benign
4                  benign
                ...      
1408214    steering_wheel
1408215    steering_wheel
1408216    steering_wheel
1408217    steering_wheel
1408218    steering_wheel
Name: target, Length: 1408219, dtype: object

In [47]:
X, Y = read_cicevse('./data/parquets/cicevse_network.parquet', 'target')

In [48]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482309 entries, 0 to 482308
Data columns (total 67 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   bidirectional_first_seen_ms   482309 non-null  int64  
 1   bidirectional_last_seen_ms    482309 non-null  int64  
 2   bidirectional_duration_ms     482309 non-null  int64  
 3   bidirectional_packets         482309 non-null  int64  
 4   bidirectional_bytes           482309 non-null  int64  
 5   src2dst_first_seen_ms         482309 non-null  int64  
 6   src2dst_last_seen_ms          482309 non-null  int64  
 7   src2dst_duration_ms           482309 non-null  int64  
 8   src2dst_packets               482309 non-null  int64  
 9   src2dst_bytes                 482309 non-null  int64  
 10  dst2src_first_seen_ms         482309 non-null  int64  
 11  dst2src_last_seen_ms          482309 non-null  int64  
 12  dst2src_duration_ms           482309 non-nul

In [3]:
results = []
raw_results = {}
for file in os.listdir('results'):
    if file.endswith('.json'):
        file_path = os.path.join('results', file)
        with open(file_path, 'r') as f:
            data = json.load(f)
            results_arrays = {}
            results_arrays['Model'] = file
            results_arrays['Accuracy'] = np.mean(data['balanced_accuracy'])
            results_arrays['AUPRC'] = np.mean(data['val_AUPRC'])
            results_arrays['MCC'] = np.mean(data['val_MCC'])
            results_arrays['F1'] = np.mean(data['val_f1'])
            raw_results[file] = data
            results.append(results_arrays)
            # results_arrays['Loss'] = np.mean(data['val_loss'])

# # 3. Calculate mean of balanced_accuracy
# balanced_accuracies = [result['balanced_accuracy'] for result in results_arrays]
# mean_accuracy = np.mean(balanced_accuracies)

# print(f"Mean balanced accuracy: {mean_accuracy:.4f}")

In [5]:
baseline = 'no-polyak_euclidean-1_chebyshev-0_cosine-0_wasserstein-0.json'

In [ ]:

import numpy as np


def perform_ttest(group_a, group_b, alpha=0.05):
    mean_a = np.mean(group_a)
    mean_b = np.mean(group_b)
    std_a = np.std(group_a, ddof=1)  # ddof=1 for sample standard deviation
    std_b = np.std(group_b, ddof=1)

    t_stat, p_value = stats.ttest_ind(group_a, group_b)

    print(f"Group A - Mean: {mean_a:.4f}, Std: {std_a:.4f}")
    print(f"Group B - Mean: {mean_b:.4f}, Std: {std_b:.4f}")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    if p_value < alpha:
        print("The difference is statistically significant (p < 0.05)")
    else:
        print("The difference is not statistically significant (p >= 0.05)")


perform_ttest(raw_results[baseline]['balanced_accuracy'],
              raw_results['polyak_euclidean-1over4_chebyshev-1over4_cosine-1over4_wasserstein-1over4.json']['balanced_accuracy'])

Group A - Mean: 0.8460, Std: 0.0316
Group B - Mean: 0.8700, Std: 0.0274
t-statistic: -3.6303
p-value: 0.0005
The difference is statistically significant (p < 0.05)


In [ ]:
pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)  # 200, att

,Model,Accuracy,AUPRC,MCC,F1
15,polyak_euclidean-1over2_chebyshev-0_cosine-1ov...,0.876787,0.465844,0.634678,0.662575
1,no-polyak_euclidean-1over3_chebyshev-1over3_co...,0.873976,0.479766,0.641218,0.672710
9,polyak_euclidean-0_chebyshev-1over3_cosine-1ov...,0.872501,0.451537,0.613864,0.649868
10,polyak_euclidean-1over3_chebyshev-1over3_cosin...,0.871558,0.464902,0.629037,0.659325
11,polyak_euclidean-1over4_chebyshev-1over4_cosin...,0.870034,0.415222,0.582885,0.617276
7,no-polyak_euclidean-1over2_chebyshev-0_cosine-...,0.870002,0.500339,0.656027,0.688587
0,no-polyak_euclidean-0_chebyshev-1over3_cosine-...,0.869473,0.459097,0.624010,0.656282
3,no-polyak_euclidean-1over4_chebyshev-1over4_co...,0.866381,0.448133,0.610248,0.646599
13,polyak_euclidean-1_chebyshev-0_cosine-0_wasser...,0.857176,0.401005,0.579437,0.605207
8,no-polyak_euclidean-1_chebyshev-0_cosine-0_was...,0.846050,0.391342,0.576802,0.598169


In [ ]:
200